# Day 6: Advanced RAG — Hybrid Dense + Lexical Reranking
**Config Source:** `.env` + `configs/app_config.yaml` (`engine/settings.py`) — Zero hardcoded URLs in notebook cells.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
ACTIVE_DOMAIN = SETTINGS.active_domain  # Override here or via BOOTCAMP_DOMAIN in .env

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Two-Stage Hybrid Reranking**
Tests incident index `2` (the domain's red-herring CPU incident) by fusing Qdrant cosine similarity with exact technical token overlap.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
import re

tricky_inc = INCIDENTS[2]
dbg.start_run("day06_rerank")
with dbg.start_span(f"Day6_HybridRerank_{tricky_inc['incident_id']}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    stage1_hits = qdrant.search(tricky_inc["raw_log"], top_k=4)
    t_s1 = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("stage1_dense_qdrant", tricky_inc["raw_log"], stage1_hits, t_s1, span_kind="RETRIEVER")

    t1 = time.perf_counter()
    q_tokens = set(re.findall(r"\w+", tricky_inc["raw_log"].lower()))
    reranked = []
    for h in stage1_hits:
        d_tokens = set(re.findall(r"\w+", h["text"].lower()))
        lex_overlap = len(q_tokens & d_tokens) / max(len(q_tokens), 1)
        hybrid = round(0.55 * h["score"] + 0.45 * lex_overlap, 4)
        reranked.append({**h, "lexical_score": round(lex_overlap, 4), "hybrid_score": hybrid})
    reranked.sort(key=lambda x: x["hybrid_score"], reverse=True)
    t_s2 = round((time.perf_counter() - t1) * 1000, 2)
    dbg.record_step("stage2_hybrid_reranker", stage1_hits, reranked, t_s2, span_kind="RERANKER")

viz.render_story(
    title=f"Day 6 Hybrid Reranker — {tricky_inc['incident_id']}",
    stages=[
        {"name": "Stage 1: Dense Candidates", "badge": "BI-ENCODER", "latency_ms": t_s1, "summary": "Cosine top-4", "scores": [{"label": h["metadata"]["runbook_id"], "score": h["score"]} for h in stage1_hits], "payload": [h["metadata"]["runbook_id"] for h in stage1_hits]},
        {"name": "Stage 2: Hybrid Reranked Order", "badge": "RERANKER", "latency_ms": t_s2, "summary": f"Top match: {reranked[0]['metadata']['runbook_id']}", "scores": [{"label": r["metadata"]["runbook_id"], "score": r["hybrid_score"]} for r in reranked], "payload": reranked[0]}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI Ranking API active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local Hybrid Reranker verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert reranked[0]["metadata"]["runbook_id"] == tricky_inc["expected_runbook"]
print("[PASS] Day 6 Hybrid Reranking Gate passed!")
